# GOLD LAYER: Valuation Analysis

Identify overvalued ("bubble") stocks by comparing:
- **Valuation:** P/E ratio, Price-to-Sales
- **Growth:** Revenue growth rate
- **Profitability:** Net profit margin
- **Risk:** Bubble risk score (0-100)

In [ ]:
# Save results
export_data = analysis[['ticker', 'close_price', 'PE_Ratio', 'Revenue growth', 'Net Margin', 'Bubble_Risk_Score']].copy()
export_data.columns = ['Ticker', 'Stock_Price', 'PE_Ratio', 'Growth_Pct', 'Margin_Pct', 'Risk_Score']
export_data.to_csv('/workspaces/bubbleindex/gold/valuation_results.csv', index=False)

print("\n" + "="*70)
print("GOLD LAYER COMPLETE ✓")
print("="*70)
print("✓ Analyzed: 5 tech companies")
print("✓ Calculated: Valuation metrics (P/E, Price-to-Sales)")
print("✓ Assessed: Bubble risk (0-100 score)")
print("✓ Exported: /workspaces/bubbleindex/gold/valuation_results.csv")
print("="*70)

## 5. Export Results

In [ ]:
print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)

top_growth = analysis.nlargest(1, 'Revenue growth')
top_margin = analysis.nlargest(1, 'Net Margin')
lowest_pe = analysis.nsmallest(1, 'PE_Ratio')
highest_risk = analysis.nlargest(1, 'Bubble_Risk_Score')
lowest_risk = analysis.nsmallest(1, 'Bubble_Risk_Score')

print(f"\n✓ Fastest Grower: {top_growth['ticker'].values[0]}")
print(f"  → Growth: {top_growth['Revenue growth'].values[0]*100:.1f}%")

print(f"\n✓ Most Profitable: {top_margin['ticker'].values[0]}")
print(f"  → Net Margin: {top_margin['Net Margin'].values[0]*100:.1f}%")

print(f"\n✓ Best Value (Cheapest): {lowest_pe['ticker'].values[0]}")
print(f"  → P/E: {lowest_pe['PE_Ratio'].values[0]:.1f}x")

print(f"\n⚠️  Highest Bubble Risk: {highest_risk['ticker'].values[0]}")
print(f"  → Risk Score: {highest_risk['Bubble_Risk_Score'].values[0]:.0f}/100")

print(f"\n✅ Safest Investment: {lowest_risk['ticker'].values[0]}")
print(f"  → Risk Score: {lowest_risk['Bubble_Risk_Score'].values[0]:.0f}/100")
print("="*70)

## 4. Key Insights

In [ ]:
print("\n" + "="*70)
print("BUBBLE RISK RANKINGS (0-100 scale)")
print("="*70)

for idx, row in analysis.sort_values('Bubble_Risk_Score', ascending=False).iterrows():
    ticker = row['ticker']
    risk = row['Bubble_Risk_Score']
    status = "🔴 HIGH" if risk > 60 else "🟡 MODERATE" if risk > 40 else "🟢 LOW"
    print(f"{ticker:8}  {risk:5.0f}/100  {status}")

print("\n" + "="*70)
print("HOW IT WORKS:")
print("="*70)
print("Bubble Risk = High Valuation (P/E) + Slowing Growth")
print("• Score > 60: Overvalued & risky")
print("• Score 40-60: Fairly valued with risks")
print("• Score < 40: Good value or strong growth")
print("="*70)

## 3. Bubble Risk Rankings

In [ ]:
# Get latest fundamentals and prices for each company
latest_fundamentals = fundamentals.sort_values('Date').groupby('ticker').tail(1).reset_index(drop=True)
latest_prices = prices.sort_values('Date').groupby('ticker').tail(1).reset_index(drop=True)

# Merge latest metrics
analysis = latest_fundamentals.merge(latest_prices, on='ticker', how='inner')

# Calculate valuation multiples
analysis['PE_Ratio'] = analysis['close_price'] / analysis['Diluted EPS']
analysis['Price_to_Sales'] = (analysis['close_price'] * 1000000) / analysis['Total Revenue']

# Bubble Risk Score (0-100): High P/E + Slowing Growth
analysis['Bubble_Risk_Score'] = (
    (analysis['PE_Ratio'] / analysis['PE_Ratio'].max()) * 50 +      # 50 pts for high P/E
    (np.maximum(0, -analysis['Revenue growth']) / 0.3) * 50          # 50 pts for negative/slowing growth
).clip(0, 100)

print("\n" + "="*70)
print("VALUATION ANALYSIS - Key Metrics")
print("="*70)

results_table = analysis[['ticker', 'close_price', 'PE_Ratio', 'Revenue growth', 'Net Margin', 'Bubble_Risk_Score']].copy()
results_table.columns = ['Ticker', 'Stock Price', 'P/E Ratio', 'Growth %', 'Margin %', 'Risk Score']
print(results_table.to_string(index=False))

## 2. Calculate Valuation Metrics & Bubble Risk

In [ ]:
import pandas as pd
import numpy as np

# Load silver layer outputs
fundamentals = pd.read_csv('/workspaces/bubbleindex/silver/combined_fundamentals.csv')
prices = pd.read_csv('/workspaces/bubbleindex/silver/combined_stockprices.csv')

# Convert dates to datetime
fundamentals['Date'] = pd.to_datetime(fundamentals['Date'])
prices['Date'] = pd.to_datetime(prices['Date'])

print(f"✓ Loaded: {fundamentals['ticker'].nunique()} companies")
print(f"✓ Fundamentals: {len(fundamentals)} rows")
print(f"✓ Prices: {len(prices)} rows")

## 1. Load & Prepare Data